# ML Final Project: Ensemble Methods — Boosting vs. Bagging

This notebook provides a high-level overview of the experimental results.
The full experiments are run via `src/experiments/run_all.py`; here we
load the computed `results/summary.json` and produce summary tables
and figures for quick inspection.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import os

# Load results
with open(os.path.join('..', 'results', 'summary.json')) as f:
    results = json.load(f)

print('Datasets:', list(results.keys()))
for ds_name, ds in results.items():
    print(f'  {ds_name}: {len(ds)} experiments')

## 1. Baseline Comparison

Verifies our from-scratch Decision Tree matches sklearn's implementation within 2% accuracy.

In [ ]:
print('Baseline accuracy gaps (our vs sklearn):')
for ds_name, ds in results.items():
    b = ds['exp1_baseline']
    gap = b['our_vs_sklearn_accuracy_gap']
    ok = '✓' if b['within_2pct'] else '✗'
    print(f'  {ds_name:20s}: gap={gap:.5f}  {ok}')

## 2. Head-to-Head Comparison (5-fold CV)

In [ ]:
print(f'{"Dataset":20s} {"Model":15s} {"Accuracy":>10s} {"F1 Macro":>10s} {"AUC-ROC":>10s}')
print('-' * 70)
for ds_name, ds in results.items():
    h2h = ds['exp4_head_to_head']
    for model_name in ['single_tree', 'adaboost', 'random_forest', 'sklearn_rf']:
        acc = h2h[model_name]['accuracy']
        f1 = h2h[model_name]['f1_macro']
        auc = h2h[model_name]['auc_roc']
        acc_str = f'{acc[0]:.4f}±{acc[1]:.4f}'
        f1_str = f'{f1[0]:.4f}±{f1[1]:.4f}'
        auc_str = f'{auc[0]:.4f}±{auc[1]:.4f}'
        print(f'{ds_name:20s} {model_name:15s} {acc_str:>10s} {f1_str:>10s} {auc_str:>10s}')
    print()

## 3. Noise Robustness

In [ ]:
print(f'{"Dataset":20s} {"Model":15s} {"5% noise":>10s} {"10% noise":>10s} {"20% noise":>10s} {"Sensitive":>10s}')
print('-' * 75)
for ds_name, ds in results.items():
    nr = ds['exp5_noise_robustness']
    print(f'{ds_name:20s} {"AdaBoost":15s} {nr["adaboost_acc"][0]:10.4f} {nr["adaboost_acc"][1]:10.4f} {nr["adaboost_acc"][2]:10.4f} {"":>10s}')
    print(f'{ds_name:20s} {"RF":15s} {nr["random_forest_acc"][0]:10.4f} {nr["random_forest_acc"][1]:10.4f} {nr["random_forest_acc"][2]:10.4f} {nr["more_sensitive_to_noise"]:>10s}')
    print()

## 4. Bias-Variance Decomposition

In [ ]:
print(f'{"Dataset":20s} {"Model":15s} {"Bias²":>10s} {"Variance":>10s} {"Avg Error":>10s}')
print('-' * 65)
for ds_name, ds in results.items():
    bv = ds['exp6_bias_variance']
    for model_name in ['single_tree', 'adaboost', 'random_forest']:
        m = bv[model_name]
        print(f'{ds_name:20s} {model_name:15s} {m["bias_sq"]:10.4f} {m["variance"]:10.4f} {m["avg_error"]:10.4f}')
    print()

## 5. Unsupervised Analysis

In [ ]:
print(f'{"Dataset":20s} {"PCA 90% comps":>15s} {"K-Means k":>10s} {"K-Means ARI":>15s} {"DBSCAN eps":>12s} {"DBSCAN ARI":>15s} {"Noise frac":>12s}')
print('-' * 100)
for ds_name, ds in results.items():
    u = ds['exp7_unsupervised']
    print(f'{ds_name:20s} {u["n_components_for_90pct_variance"]:>15d} {u["kmeans_best_k"]:>10d} {u["kmeans_ari"]:>15.4f} {u["dbscan_eps"]:>12.2f} {u["dbscan_ari"]:>15.4f} {u["dbscan_noise_fraction"]:>12.4f}')

## 6. Scaling Curves

Figures are saved under `figures/<dataset>/`. Below we display one example.

In [ ]:
import matplotlib.image as mpimg

fig_dir = os.path.join('..', 'figures', 'breast_cancer')
if os.path.exists(fig_dir):
    files = [f for f in os.listdir(fig_dir) if f.endswith('.png')]
    for f in sorted(files):
        print(f'{f}')
else:
    print('No figures found. Run experiments first.')

## Key Takeaways

- **AdaBoost** excels on clean, well-separated data (MNIST, Breast Cancer).
- **Random Forest** is more robust against label noise and multi-class problems (Covertype).
- AdaBoost achieves lower bias but higher variance; Random Forest reduces variance at a slight bias cost.
- On severely imbalanced data (Adult Income), both methods achieve high accuracy but struggle with F1 macro.